# BERT Fake News Detection — SPST Training
Progressive unfreezing, mixed precision, Drive checkpointing, full evaluation plots.

In [ ]:
# ── 0. Install dependencies ───────────────────────────────────────────────────
!pip install transformers==4.40.0 scikit-learn matplotlib seaborn -q

In [ ]:
# ── 1. Mount Drive & set paths ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# ╔══════════════════════════════════════════════════╗
# ║  CONFIG — edit these if your paths change        ║
# ╚══════════════════════════════════════════════════╝
DRIVE_ROOT      = '/content/drive/MyDrive/spst'
DATASET_ZIP_ID  = '1BfIvNcVJQIx8-KAqWFavy-htKcjTbMTb'   # your Drive file ID
DATASET_DIR     = '/content/datasets'
CHECKPOINT_DIR  = os.path.join(DRIVE_ROOT, 'checkpoints')
PLOTS_DIR       = os.path.join(DRIVE_ROOT, 'plots')
FINAL_MODEL_DIR = os.path.join(DRIVE_ROOT, 'final_model')

for d in [CHECKPOINT_DIR, PLOTS_DIR, FINAL_MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted. Directories ready.')

In [ ]:
# ── 2. Download & unzip dataset from Drive ────────────────────────────────────
import gdown

zip_path = '/content/datasets.zip'
gdown.download(id=DATASET_ZIP_ID, output=zip_path, quiet=False)

!unzip -q {zip_path} -d /content/

# Locate the processed TSVs
TRAIN_TSV = '/content/datasets/processed/unified_train.tsv'
VAL_TSV   = '/content/datasets/processed/unified_val.tsv'
TEST_TSV  = '/content/datasets/processed/unified_test.tsv'

for p in [TRAIN_TSV, VAL_TSV, TEST_TSV]:
    assert os.path.exists(p), f'Missing: {p}'
    print(f'Found: {p}')

In [ ]:
# ── 3. Imports & config ───────────────────────────────────────────────────────
import gc
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from datetime import datetime
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, confusion_matrix, roc_curve
)
from tqdm.notebook import tqdm

# ── Training hyperparameters ──────────────────────────────────────────────────
CFG = {
    'model_name'                : 'bert-base-uncased',
    'num_labels'                : 2,
    'max_length'                : 128,
    'batch_size'                : 16,
    'gradient_accumulation_steps': 4,     # effective batch = 64
    'weight_decay'              : 0.01,
    'warmup_steps'              : 100,
    'max_grad_norm'             : 1.0,
    'use_mixed_precision'       : True,
    'use_gradient_checkpointing': True,
    'empty_cache_every'         : 50,     # steps
    'random_seed'               : 42,
}

# ── Progressive unfreezing segments ──────────────────────────────────────────
# Each segment unfreezes more layers, trains for N epochs at a given LR
SEGMENTS = [
    {
        'name'           : 'classifier_only',
        'layers_to_train': ['classifier', 'pooler'],
        'epochs'         : 2,
        'learning_rate'  : 3e-4,
    },
    {
        'name'           : 'top_layers',
        'layers_to_train': ['classifier', 'pooler', 'encoder.layer.11', 'encoder.layer.10'],
        'epochs'         : 2,
        'learning_rate'  : 1e-4,
    },
    {
        'name'           : 'mid_layers',
        'layers_to_train': ['classifier', 'pooler', 'encoder.layer.11', 'encoder.layer.10',
                            'encoder.layer.9', 'encoder.layer.8', 'encoder.layer.7'],
        'epochs'         : 2,
        'learning_rate'  : 5e-5,
    },
    {
        'name'           : 'full_model',
        'layers_to_train': None,   # None = unfreeze all
        'epochs'         : 2,
        'learning_rate'  : 2e-5,
    },
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── 4. Dataset & DataLoaders ──────────────────────────────────────────────────
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'labels'        : torch.tensor(self.labels[idx], dtype=torch.long)
        }


def load_split(path):
    df = pd.read_csv(path, sep='\t')
    df = df.dropna(subset=['text', 'label'])
    df['label'] = pd.to_numeric(df['label'], errors='coerce').dropna().astype(int)
    df = df.dropna(subset=['label'])
    print(f'{os.path.basename(path)}: {len(df)} rows | '
          f'fake={( df["label"]==0).sum()} real={(df["label"]==1).sum()}')
    return df


print('Loading datasets...')
train_df = load_split(TRAIN_TSV)
val_df   = load_split(VAL_TSV)
test_df  = load_split(TEST_TSV)

tokenizer = BertTokenizer.from_pretrained(CFG['model_name'])

def make_loader(df, shuffle):
    ds = FakeNewsDataset(
        df['text'].values, df['label'].values,
        tokenizer, CFG['max_length']
    )
    return DataLoader(ds, batch_size=CFG['batch_size'], shuffle=shuffle,
                      num_workers=2, pin_memory=True)

train_loader = make_loader(train_df, shuffle=True)
val_loader   = make_loader(val_df,   shuffle=False)
test_loader  = make_loader(test_df,  shuffle=False)
print('DataLoaders ready.')

In [ ]:
# ── 5. Model ──────────────────────────────────────────────────────────────────
model = BertForSequenceClassification.from_pretrained(
    CFG['model_name'], num_labels=CFG['num_labels']
)
if CFG['use_gradient_checkpointing']:
    model.gradient_checkpointing_enable()
    print('Gradient checkpointing enabled')
model.to(device)
print('Model loaded.')

In [ ]:
# ── 6. Helpers ────────────────────────────────────────────────────────────────
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def freeze_layers(model, layers_to_train):
    for param in model.parameters():
        param.requires_grad = False
    if layers_to_train is None:
        for param in model.parameters():
            param.requires_grad = True
        print('Unfrozen: ALL layers')
        return
    for name, param in model.named_parameters():
        if any(l in name for l in layers_to_train):
            param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')


def compute_metrics(preds, labels, probs=None):
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    m = {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}
    if probs is not None:
        try:
            m['auc'] = roc_auc_score(labels, probs[:, 1])
        except Exception:
            pass
    return m


def train_epoch(model, loader, optimizer, scheduler, scaler, epoch_num):
    model.train()
    total_loss, preds_all, labels_all = 0, [], []
    bar = tqdm(loader, desc=f'Train Epoch {epoch_num}')
    for step, batch in enumerate(bar):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['labels'].to(device)
        optimizer.zero_grad()
        if CFG['use_mixed_precision']:
            with autocast():
                out  = model(input_ids=ids, attention_mask=mask, labels=labs)
                loss = out.loss / CFG['gradient_accumulation_steps']
            scaler.scale(loss).backward()
        else:
            out  = model(input_ids=ids, attention_mask=mask, labels=labs)
            loss = out.loss / CFG['gradient_accumulation_steps']
            loss.backward()
        if (step + 1) % CFG['gradient_accumulation_steps'] == 0:
            if CFG['use_mixed_precision']:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['max_grad_norm'])
            if CFG['use_mixed_precision']:
                scaler.step(optimizer); scaler.update()
            else:
                optimizer.step()
            scheduler.step()
        total_loss += loss.item() * CFG['gradient_accumulation_steps']
        preds_all.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
        labels_all.extend(labs.cpu().numpy())
        if step % CFG['empty_cache_every'] == 0:
            clear_memory()
        bar.set_postfix(loss=f"{loss.item()*CFG['gradient_accumulation_steps']:.4f}")
    m = compute_metrics(preds_all, labels_all)
    m['loss'] = total_loss / len(loader)
    clear_memory()
    return m


def evaluate(model, loader, desc='Eval'):
    model.eval()
    total_loss, preds_all, labels_all, probs_all = 0, [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labs = batch['labels'].to(device)
            if CFG['use_mixed_precision']:
                with autocast():
                    out = model(input_ids=ids, attention_mask=mask, labels=labs)
            else:
                out = model(input_ids=ids, attention_mask=mask, labels=labs)
            total_loss += out.loss.item()
            probs = torch.softmax(out.logits, dim=1)
            preds_all.extend(torch.argmax(probs, dim=1).cpu().numpy())
            labels_all.extend(labs.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())
    probs_all = np.array(probs_all)
    m = compute_metrics(preds_all, labels_all, probs_all)
    m['loss'] = total_loss / len(loader)
    m['_preds']  = preds_all
    m['_labels'] = labels_all
    m['_probs']  = probs_all
    clear_memory()
    return m


def save_checkpoint(model, tokenizer, optimizer, segment_name, epoch, metrics):
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'{segment_name}_epoch{epoch}')
    os.makedirs(ckpt_path, exist_ok=True)
    model.save_pretrained(ckpt_path)
    tokenizer.save_pretrained(ckpt_path)
    torch.save({'epoch': epoch, 'segment': segment_name,
                'optimizer': optimizer.state_dict(), 'metrics': metrics},
               os.path.join(ckpt_path, 'state.pt'))
    print(f'  Checkpoint saved -> {ckpt_path}')

print('Helpers defined.')

In [ ]:
# ── 7. Progressive Unfreezing Training ───────────────────────────────────────
history = []   # list of dicts, one per epoch across all segments
scaler  = GradScaler() if CFG['use_mixed_precision'] else None

for seg_idx, seg in enumerate(SEGMENTS):
    print(f"\n{'='*70}")
    print(f"Segment {seg_idx+1}/{len(SEGMENTS)}: {seg['name']}")
    print(f"{'='*70}")

    freeze_layers(model, seg['layers_to_train'])

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=seg['learning_rate'],
                      weight_decay=CFG['weight_decay'])
    total_steps = len(train_loader) * seg['epochs']
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=CFG['warmup_steps'] // len(SEGMENTS),
        num_training_steps=total_steps
    )

    best_f1 = 0
    for epoch in range(1, seg['epochs'] + 1):
        print(f"\n--- {seg['name']} | Epoch {epoch}/{seg['epochs']} ---")
        train_m = train_epoch(model, train_loader, optimizer, scheduler, scaler, epoch)
        val_m   = evaluate(model, val_loader, desc='Validation')

        row = {
            'segment'  : seg['name'],
            'epoch'    : epoch,
            'train_loss': train_m['loss'],  'train_f1': train_m['f1'],
            'train_acc' : train_m['accuracy'],
            'val_loss'  : val_m['loss'],    'val_f1'  : val_m['f1'],
            'val_acc'   : val_m['accuracy'], 'val_auc' : val_m.get('auc', None),
        }
        history.append(row)

        print(f"  Train  loss={train_m['loss']:.4f}  f1={train_m['f1']:.4f}  acc={train_m['accuracy']:.4f}")
        print(f"  Val    loss={val_m['loss']:.4f}  f1={val_m['f1']:.4f}  acc={val_m['accuracy']:.4f}  auc={val_m.get('auc','n/a')}")

        if val_m['f1'] > best_f1:
            best_f1 = val_m['f1']
            save_checkpoint(model, tokenizer, optimizer, seg['name'], epoch, val_m)
            print(f"  ✓ Best so far for {seg['name']}: F1={best_f1:.4f}")

print("\nProgressive unfreezing complete.")

In [ ]:
# ── 8. Final evaluation on test set ──────────────────────────────────────────
print('Running final test evaluation...')
test_m = evaluate(model, test_loader, desc='Test')

print(f"\nTest Results:")
print(f"  Accuracy  : {test_m['accuracy']:.4f}")
print(f"  Precision : {test_m['precision']:.4f}")
print(f"  Recall    : {test_m['recall']:.4f}")
print(f"  F1        : {test_m['f1']:.4f}")
print(f"  AUC       : {test_m.get('auc', 'n/a')}")

# Save final model
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f'Final model saved -> {FINAL_MODEL_DIR}')

# Save results JSON
results = {
    'test': {k: float(v) for k, v in test_m.items() if not k.startswith('_')},
    'history': history,
    'config': CFG
}
results_path = os.path.join(DRIVE_ROOT, 'results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved -> {results_path}')

In [ ]:
# ── 9. Evaluation Plots ───────────────────────────────────────────────────────
hist_df = pd.DataFrame(history)

# Helper: add segment boundary lines
def add_segment_boundaries(ax, hist_df):
    seg_ends = hist_df.groupby('segment', sort=False).apply(len).cumsum().values[:-1]
    for x in seg_ends:
        ax.axvline(x=x - 0.5, color='grey', linestyle='--', alpha=0.5)

steps = range(len(hist_df))
xlabels = [f"{r['segment']}\ne{r['epoch']}" for _, r in hist_df.iterrows()]

# ── Plot 1: Loss curves ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(steps, hist_df['train_loss'], label='Train Loss', marker='o')
ax.plot(steps, hist_df['val_loss'],   label='Val Loss',   marker='s')
ax.set_xticks(list(steps)); ax.set_xticklabels(xlabels, fontsize=7, rotation=45)
add_segment_boundaries(ax, hist_df)
ax.set_title('Loss Curves across Segments'); ax.set_ylabel('Loss')
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'loss_curves.png'), dpi=150)
plt.show(); plt.close()

# ── Plot 2: F1 curves ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(steps, hist_df['train_f1'], label='Train F1', marker='o')
ax.plot(steps, hist_df['val_f1'],   label='Val F1',   marker='s')
ax.set_xticks(list(steps)); ax.set_xticklabels(xlabels, fontsize=7, rotation=45)
add_segment_boundaries(ax, hist_df)
ax.set_title('F1 Score across Segments'); ax.set_ylabel('F1')
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'f1_curves.png'), dpi=150)
plt.show(); plt.close()

# ── Plot 3: Accuracy curves ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(steps, hist_df['train_acc'], label='Train Acc', marker='o')
ax.plot(steps, hist_df['val_acc'],   label='Val Acc',   marker='s')
ax.set_xticks(list(steps)); ax.set_xticklabels(xlabels, fontsize=7, rotation=45)
add_segment_boundaries(ax, hist_df)
ax.set_title('Accuracy across Segments'); ax.set_ylabel('Accuracy')
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'accuracy_curves.png'), dpi=150)
plt.show(); plt.close()

# ── Plot 4: Val AUC per segment ───────────────────────────────────────────────
if hist_df['val_auc'].notna().any():
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(steps, hist_df['val_auc'], label='Val AUC', marker='D', color='purple')
    ax.set_xticks(list(steps)); ax.set_xticklabels(xlabels, fontsize=7, rotation=45)
    add_segment_boundaries(ax, hist_df)
    ax.set_title('Validation AUC across Segments'); ax.set_ylabel('AUC')
    ax.legend(); plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'auc_curves.png'), dpi=150)
    plt.show(); plt.close()

# ── Plot 5: Confusion matrix (test set) ──────────────────────────────────────
cm = confusion_matrix(test_m['_labels'], test_m['_preds'])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show(); plt.close()

# ── Plot 6: ROC curve (test set) ──────────────────────────────────────────────
if 'auc' in test_m:
    fpr, tpr, _ = roc_curve(test_m['_labels'], test_m['_probs'][:, 1])
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fpr, tpr, label=f"AUC = {test_m['auc']:.4f}")
    ax.plot([0,1],[0,1],'k--', alpha=0.4)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curve — Test Set')
    ax.legend(); plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'roc_curve.png'), dpi=150)
    plt.show(); plt.close()

# ── Plot 7: Per-segment best val F1 bar chart ─────────────────────────────────
seg_best = hist_df.groupby('segment', sort=False)['val_f1'].max().reset_index()
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(seg_best['segment'], seg_best['val_f1'], color='steelblue')
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.set_ylim(0, 1.05); ax.set_ylabel('Best Val F1')
ax.set_title('Best Validation F1 per Training Segment')
plt.xticks(rotation=15); plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'segment_f1_bar.png'), dpi=150)
plt.show(); plt.close()

print(f'All plots saved to {PLOTS_DIR}')

In [ ]:
# ── 10. Download plots & results locally ──────────────────────────────────────
from google.colab import files
import zipfile

zip_out = '/content/spst_results.zip'
with zipfile.ZipFile(zip_out, 'w') as zf:
    for fname in os.listdir(PLOTS_DIR):
        zf.write(os.path.join(PLOTS_DIR, fname), fname)
    zf.write(results_path, 'results.json')

files.download(zip_out)
print('Download triggered: spst_results.zip')